# 2. Results: performance, errors and reliability

Reads the outputs of `evaluate` and `compare`. All choices (epoch, temperature, referral threshold, quality bands, primary model) were made on training or validation data; the test split is only measured.

In [ ]:
import json, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

from drxai import GRADE_NAMES
from drxai.config import REPO_ROOT, cache_dir, load_config, resolve

# Read-only: this notebook loads files written by the pipeline commands and computes nothing that is reported elsewhere.
cfg = load_config(os.environ.get("DRXAI_CONFIG", REPO_ROOT / "configs" / "default.yaml"))
CACHE = cache_dir(cfg)
RUNS = resolve(cfg["paths"]["runs_dir"])

In [ ]:
comparison = json.loads((RUNS / "comparison.json").read_text())
PRIMARY = Path(comparison["primary"])
print("primary run (chosen on validation QWK):", PRIMARY.name)

## Model choice: chosen on validation, never on test

In [ ]:
display(pd.DataFrame(comparison["runs"]).set_index("run"))
d = comparison["test_qwk_difference_primary_minus_other"]
print(f"test QWK difference (primary minus other): {d['point']:+.3f}, 95% CI {d['lo']:+.3f} to {d['hi']:+.3f}")
print("interval excludes zero" if comparison["interval_excludes_zero"] else "interval includes zero: no demonstrated difference")

## Headline metrics against the majority-class floor

In [ ]:
m = json.loads((PRIMARY / "metrics.json").read_text())
rows = []
for k, name in [("qwk", "Quadratic weighted kappa"), ("macro_f1", "Macro-F1"), ("accuracy", "Accuracy")]:
    ci = m["bootstrap_95ci"][k]
    rows.append({"metric": name, "model": m["performance"][k], "95% CI": f"{ci['lo']:.3f} to {ci['hi']:.3f}",
                 "floor": m["majority_floor"][k], "margin": m["margin_above_floor"][k]})
pd.DataFrame(rows).set_index("metric").round(3)

In [ ]:
pc = pd.DataFrame(m["performance"]["per_class"]).T
pc.index = [f"{int(g)} {GRADE_NAMES[int(g)]}" for g in pc.index]
display(pc.round(3))
display(Image(str(PRIMARY / "figures" / "confusion_matrix.png"), width=520))

## Error direction
Under-grading is the clinically costly direction for triage: it gives a patient less optometrist time than they need.

In [ ]:
e = m["error_direction"]
pd.Series({"errors": e["n_errors"], "share under-graded": e["share_of_errors_under"],
           "share over-graded": e["share_of_errors_over"], "two or more grades out": e["errors_two_or_more_grades"],
           "missed referable (grade >= 2 called < 2)": e["missed_referable"],
           "referable sensitivity": e["referable_sensitivity"], "referable specificity": e["referable_specificity"],
           "referable AUROC": m["referable_auc"]}).round(3).to_frame("value")

## Reliability: calibration
Temperature fitted on validation, assessed on test.

In [ ]:
cal = m["calibration"]
print(f"temperature {m['temperature']:.3f} | ECE uncalibrated {cal['ece_uncalibrated']['ece']:.3f} -> scaled {cal['ece_temperature_scaled']['ece']:.3f}")
display(Image(str(PRIMARY / "figures" / "reliability.png"), width=420))

## Reliability: referral rule for triage
Threshold fixed on validation for 80% automatic coverage, then applied to test. Compared with referring the same number of cases at random.

In [ ]:
display(pd.Series(m["referral_rule"]).to_frame("value"))
display(pd.DataFrame(m["referral_curve"]).set_index("coverage").round(3))
display(Image(str(PRIMARY / "figures" / "referral_curve.png"), width=760))

## Reliability: image quality
Real sharpness bands (edges from training images), and simulated blur or under-exposure if `--quality` was run. The dangerous pattern is accuracy falling while confidence stays high.

In [ ]:
if "quality_strata_sharpness" in m:
    q = m["quality_strata_sharpness"]
    display(pd.DataFrame(q["bands"]).set_index("band").round(3))
    print(f"Spearman sharpness vs confidence {q['spearman_quality_vs_confidence']:.3f}; vs correct {q['spearman_quality_vs_correct']:.3f}")
if "quality_probe" in m:
    display(pd.DataFrame(m["quality_probe"]).set_index("condition").round(3))
    display(Image(str(PRIMARY / "figures" / "quality_probe.png"), width=600))